# 13.10. 转置卷积

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

输入矩阵X和卷积核矩阵K

In [2]:
def trans_conv( X , K ):
    h ,w  =K.shape
    Y = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1)) # 忘了这个语法了
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Y[i:i + h, j:j + w] += X[i, j] * K
    return Y

In [3]:
X = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
trans_conv(X, K)

tensor([[ 0.,  0.,  1.],
        [ 0.,  4.,  6.],
        [ 4., 12.,  9.]])

- (batch_size, channels, height, width)
- nn.ConvTranspose2d(
    in_channels=1,
    out_channels=1,
    kernel_size=2,
    bias=False
)

In [ ]:
X, K = X.reshape(1, 1, 2, 2), K.reshape(1, 1, 2, 2)
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False)
tconv.weight.data = K  # 这什么意思
tconv(X)

tensor([[[[ 0.,  0.,  1.],
          [ 0.,  4.,  6.],
          [ 4., 12.,  9.]]]], grad_fn=<ConvolutionBackward0>)

## 13.10.2. 填充、步幅和多通道

- 在转置卷积中，填充被应用于的输出
- 当将高和宽两侧的填充数指定为1时，转置卷积的输出中将删除第一和最后的行与列。

In [ ]:
tconv = nn.ConvTranspose2d(1,1,kernel_size= 2,padding =1,bias =False)  # 前两个参数是什么来着
tconv.weight.data = K
tconv(X)

tensor([[[[4.]]]], grad_fn=<ConvolutionBackward0>)

- 步幅被指定为中间结果（输出），而不是输入

In [6]:
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
tconv.weight.data = K
tconv(X)

tensor([[[[0., 0., 0., 1.],
          [0., 0., 2., 3.],
          [0., 2., 0., 3.],
          [4., 6., 6., 9.]]]], grad_fn=<ConvolutionBackward0>)

- 对于多个输入和输出通道，转置卷积与常规卷积以相同方式运作。
- 这个有点不明白，首先是这邪恶数字都代表什么

In [7]:
X = torch.rand(size=(1, 10, 16, 16))
conv = nn.Conv2d(10, 20, kernel_size=5, padding=2, stride=3)
tconv = nn.ConvTranspose2d(20, 10, kernel_size=5, padding=2, stride=3)
tconv(conv(X)).shape == X.shape

True

## 13.10.3. 与矩阵变换的联系

In [8]:
X =torch.arange(9.0).reshape(3,3)
K =torch.tensor([[1.0, 2.0], [3.0, 4.0]])
Y = d2l.corr2d(X, K)
Y

tensor([[27., 37.],
        [57., 67.]])

- 把原本的卷积计算，转换成矩阵乘法。
- 为什么 `W `是 `4×9`。
  - `X.numel()  # 9`
  - `Y.numel()  # 4`
  - `Y.flatten() = W @ X.flatten()`

- 

In [10]:
def kernel2matrix(K):
    k, W = torch.zeros(5), torch.zeros((4, 9))
    k[:2], k[3:5] = K[0, :], K[1, :]
    W[0, :5], W[1, 1:6], W[2, 3:8], W[3, 4:] = k, k, k, k
    return W

W = kernel2matrix(K)
W

tensor([[1., 2., 0., 3., 4., 0., 0., 0., 0.],
        [0., 1., 2., 0., 3., 4., 0., 0., 0.],
        [0., 0., 0., 1., 2., 0., 3., 4., 0.],
        [0., 0., 0., 0., 1., 2., 0., 3., 4.]])

In [15]:
Y == torch.matmul(W, X.reshape(-1)).reshape(2, 2)


tensor([[True, True],
        [True, True]])

- 使用矩阵乘法来实现转置卷积

In [17]:
Z = trans_conv(Y, K)
Z == torch.matmul(W.T, Y.reshape(-1)).reshape(3, 3)
Z,X

(tensor([[ 27.,  91.,  74.],
         [138., 400., 282.],
         [171., 429., 268.]]),
 tensor([[0., 1., 2.],
         [3., 4., 5.],
         [6., 7., 8.]]))

- 看到这，我想我明白我哪里疑惑了，是对卷积，矩阵乘法的原理忘了，计算符号也忘了。